# 📡 Notebook 2: Pub/Sub and Streams

Redis supports two messaging paradigms: **Pub/Sub** for fire-and-forget real-time broadcasts, and **Streams** for durable, replayable event logs with consumer groups. Understanding when to use each is critical for system design.

## Learning Objectives
- Understand Redis Pub/Sub: channels, publishing, subscribing
- See why Pub/Sub is "fire-and-forget" and its limitations
- Learn Redis Streams for durable messaging
- Implement consumer groups for distributed work queues
- Know when to choose Pub/Sub vs Streams vs Kafka

## 🛠️ Setup

```bash
cd 03-technologies/databases/redis
docker compose up -d
```

### Visualization
- **RedisInsight**: http://localhost:5540 — connect to `redis://localhost:6379`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".

In [1]:
import redis
import threading
import time
import json

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
    r.flushdb()
    print("🧹 Flushed database for a clean start")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")
    print("   Run: cd 03-technologies/databases/redis && docker compose up -d")

✅ Connected to Redis
🧹 Flushed database for a clean start


## 1️⃣ Redis Pub/Sub

Pub/Sub (Publish/Subscribe) lets you broadcast messages to multiple listeners in real time. Think of it like a radio station: anyone tuned in hears the message, but if you're not listening, you miss it.

```
Publisher → PUBLISH channel message
                    ↓
            ┌───────────────┐
            │   Channel     │
            └───┬───┬───┬───┘
                ↓   ↓   ↓
            Sub1  Sub2  Sub3
```

Key characteristics:
- **Fire-and-forget**: messages are NOT stored. If no one is listening, the message is lost.
- **At-most-once delivery**: a subscriber receives a message at most once.
- **Real-time only**: you cannot replay old messages.

In [2]:
# Pub/Sub requires a dedicated connection for the subscriber
# The subscriber blocks while listening, so we run it in a thread

received_messages = []

def subscriber_worker(channel_name, message_limit=3):
    """Listens on a channel and collects messages."""
    sub_client = redis.Redis(host="localhost", port=6379, decode_responses=True)
    pubsub = sub_client.pubsub()
    pubsub.subscribe(channel_name)
    
    count = 0
    for message in pubsub.listen():
        if message["type"] == "message":
            received_messages.append(message["data"])
            print(f"  📨 Subscriber received: {message['data']}")
            count += 1
            if count >= message_limit:
                break
    
    pubsub.unsubscribe()
    pubsub.close()

# Start subscriber in a background thread
thread = threading.Thread(target=subscriber_worker, args=("notifications", 3))
thread.start()

# Give the subscriber a moment to connect
time.sleep(0.5)

# Publish messages
print("Publishing messages to 'notifications' channel:")
for i in range(3):
    msg = f"Alert #{i+1}: Something happened!"
    r.publish("notifications", msg)
    print(f"  📤 Published: {msg}")
    time.sleep(0.2)

thread.join(timeout=5)
print(f"\n✅ Subscriber received {len(received_messages)} messages")
print("\n💡 The subscriber had to be listening BEFORE messages were published!")

Publishing messages to 'notifications' channel:
  📨 Subscriber received: Alert #1: Something happened!
  📤 Published: Alert #1: Something happened!


  📨 Subscriber received: Alert #2: Something happened!  📤 Published: Alert #2: Something happened!



  📤 Published: Alert #3: Something happened!  📨 Subscriber received: Alert #3: Something happened!




✅ Subscriber received 3 messages

💡 The subscriber had to be listening BEFORE messages were published!


## ⚠️ Pub/Sub Limitation: Lost Messages

What happens if we publish a message but nobody is subscribed?

In [3]:
# Publish with NO subscribers — the message vanishes!
listeners = r.publish("ghost_channel", "Is anyone there?")
print(f"Listeners who received the message: {listeners}")
print("💡 Zero listeners = message is gone forever. Pub/Sub does NOT store messages!")

# This is why Pub/Sub is called 'fire-and-forget'
print("\n⚠️  Pub/Sub is great for real-time notifications (chat, live updates)")
print("   but NOT suitable when you need guaranteed delivery.")
print("   For durability, use Redis Streams (next section) or Kafka.")

Listeners who received the message: 0
💡 Zero listeners = message is gone forever. Pub/Sub does NOT store messages!

⚠️  Pub/Sub is great for real-time notifications (chat, live updates)
   but NOT suitable when you need guaranteed delivery.
   For durability, use Redis Streams (next section) or Kafka.


## 📺 Multiple Channels and Pattern Subscribe

You can subscribe to multiple channels, or use patterns to subscribe to channel families.

In [4]:
# Pattern subscribe example — subscribe to all "news:*" channels
pattern_messages = []

def pattern_subscriber(pattern, message_limit=4):
    """Subscribe using a pattern — matches multiple channels."""
    sub_client = redis.Redis(host="localhost", port=6379, decode_responses=True)
    pubsub = sub_client.pubsub()
    pubsub.psubscribe(pattern)
    
    count = 0
    for message in pubsub.listen():
        if message["type"] == "pmessage":
            pattern_messages.append({
                "channel": message["channel"],
                "data": message["data"]
            })
            print(f"  📨 [{message['channel']}] {message['data']}")
            count += 1
            if count >= message_limit:
                break
    
    pubsub.punsubscribe()
    pubsub.close()

# Subscribe to all channels matching "news:*"
thread = threading.Thread(target=pattern_subscriber, args=("news:*", 4))
thread.start()
time.sleep(0.5)

# Publish to different news channels
r.publish("news:sports", "Team wins championship!")
r.publish("news:tech", "New AI model released")
r.publish("news:weather", "Sunny skies tomorrow")
r.publish("news:sports", "Player breaks record")
time.sleep(0.2)

# A publish to a non-matching channel is NOT received
r.publish("alerts:security", "This won't be received")

thread.join(timeout=5)
print(f"\n✅ Pattern subscriber received {len(pattern_messages)} messages from multiple channels")
print("💡 PSUBSCRIBE 'news:*' catches all channels starting with 'news:'")

  📨 [news:sports] Team wins championship!
  📨 [news:tech] New AI model released
  📨 [news:weather] Sunny skies tomorrow
  📨 [news:sports] Player breaks record



✅ Pattern subscriber received 4 messages from multiple channels
💡 PSUBSCRIBE 'news:*' catches all channels starting with 'news:'


## 2️⃣ Redis Streams

Streams solve the limitations of Pub/Sub. They are **append-only logs** (like Kafka topics) where messages are **stored permanently** (until you delete them).

```
Pub/Sub:    Publisher → Channel → Subscriber (message gone!)
Streams:    Producer → Stream [msg1, msg2, msg3, ...] → Consumer (replay anytime!)
```

Key differences from Pub/Sub:
| Feature | Pub/Sub | Streams |
|---------|---------|----------|
| Message persistence | ❌ No | ✅ Yes |
| Replay old messages | ❌ No | ✅ Yes |
| Consumer groups | ❌ No | ✅ Yes |
| Acknowledgment | ❌ No | ✅ Yes |
| Best for | Real-time broadcasts | Durable event processing |

In [5]:
# XADD — add messages to a stream
# "*" means auto-generate an ID (timestamp-based)
r.xadd("orders", {"action": "created", "order_id": "1001", "customer": "Alice", "total": "59.99"})
time.sleep(0.01)  # Small delay so IDs are different
r.xadd("orders", {"action": "paid", "order_id": "1001", "method": "credit_card"})
time.sleep(0.01)
r.xadd("orders", {"action": "created", "order_id": "1002", "customer": "Bob", "total": "129.00"})
time.sleep(0.01)
r.xadd("orders", {"action": "shipped", "order_id": "1001", "carrier": "FedEx"})

print(f"Stream length: {r.xlen('orders')} messages")

# XRANGE — read all messages (oldest to newest)
print("\nAll order events:")
for entry_id, fields in r.xrange("orders"):
    print(f"  [{entry_id}] {fields}")

Stream length: 4 messages

All order events:
  [1776555982907-0] {'action': 'created', 'order_id': '1001', 'customer': 'Alice', 'total': '59.99'}
  [1776555982990-0] {'action': 'paid', 'order_id': '1001', 'method': 'credit_card'}
  [1776555983073-0] {'action': 'created', 'order_id': '1002', 'customer': 'Bob', 'total': '129.00'}
  [1776555983163-0] {'action': 'shipped', 'order_id': '1001', 'carrier': 'FedEx'}


### Reading from Streams

Unlike Pub/Sub, stream consumers can start reading from any point — the beginning, a specific ID, or only new messages.

In [6]:
# Read from the beginning
print("📖 Reading from the beginning (id='0'):")
messages = r.xrange("orders", min="-", max="+")
for entry_id, fields in messages:
    print(f"  [{entry_id}] action={fields['action']}, order={fields['order_id']}")

# Read only the latest message
print("\n📖 Latest message only:")
latest = r.xrevrange("orders", count=1)
for entry_id, fields in latest:
    print(f"  [{entry_id}] {fields}")

# XREAD — blocking read for new messages (like a subscription)
# This would normally block, but we can set a timeout
print("\n📖 XREAD with 1-second timeout (waiting for new messages):")
result = r.xread({"orders": "$"}, block=1000, count=1)  # "$" = only new messages
if result:
    print(f"  Got: {result}")
else:
    print("  No new messages within 1 second (expected — we didn't publish any)")

print("\n💡 Streams store messages permanently — consumers can catch up after downtime!")

📖 Reading from the beginning (id='0'):
  [1776555982907-0] action=created, order=1001
  [1776555982990-0] action=paid, order=1001
  [1776555983073-0] action=created, order=1002
  [1776555983163-0] action=shipped, order=1001

📖 Latest message only:
  [1776555983163-0] {'action': 'shipped', 'order_id': '1001', 'carrier': 'FedEx'}

📖 XREAD with 1-second timeout (waiting for new messages):


  No new messages within 1 second (expected — we didn't publish any)

💡 Streams store messages permanently — consumers can catch up after downtime!


## 3️⃣ Consumer Groups

Consumer groups allow multiple workers to **divide work** from a single stream. Each message is delivered to exactly one consumer in the group — perfect for work queues.

```
Stream: [msg1, msg2, msg3, msg4, msg5, msg6]
                  ↓ Consumer Group "processors"
         ┌───────┼───────┐
         ↓       ↓       ↓
      Worker1  Worker2  Worker3
      (msg1)   (msg2)   (msg3)
      (msg4)   (msg5)   (msg6)
```

Each consumer in a group:
- Gets unique messages (no duplicates between workers)
- Must **acknowledge** (XACK) when done processing
- If a consumer crashes, unacknowledged messages can be **claimed** (XCLAIM) by another worker

In [7]:
# Create a fresh stream with some work items
r.delete("tasks")
for i in range(6):
    r.xadd("tasks", {"task_id": f"task_{i}", "type": "process_image", "url": f"img_{i}.jpg"})

print(f"Created {r.xlen('tasks')} tasks in the stream")

# Create a consumer group starting from the beginning of the stream
# "0" means read from the very first message
try:
    r.xgroup_create("tasks", "image_processors", id="0", mkstream=True)
    print("✅ Created consumer group 'image_processors'")
except redis.exceptions.ResponseError as e:
    if "BUSYGROUP" in str(e):
        print("ℹ️  Consumer group already exists")
    else:
        raise

# Worker A reads some tasks
worker_a_tasks = r.xreadgroup(
    "image_processors", "worker_a",
    {"tasks": ">"},  # ">" means only new (undelivered) messages
    count=2
)
print("\n🔧 Worker A received:")
for stream, messages in worker_a_tasks:
    for msg_id, fields in messages:
        print(f"  [{msg_id}] {fields}")

# Worker B reads some tasks
worker_b_tasks = r.xreadgroup(
    "image_processors", "worker_b",
    {"tasks": ">"},
    count=2
)
print("\n🔧 Worker B received:")
for stream, messages in worker_b_tasks:
    for msg_id, fields in messages:
        print(f"  [{msg_id}] {fields}")

# Worker C reads remaining tasks
worker_c_tasks = r.xreadgroup(
    "image_processors", "worker_c",
    {"tasks": ">"},
    count=2
)
print("\n🔧 Worker C received:")
for stream, messages in worker_c_tasks:
    for msg_id, fields in messages:
        print(f"  [{msg_id}] {fields}")

print("\n💡 Each task was delivered to exactly ONE worker — no duplicates!")

Created 6 tasks in the stream
✅ Created consumer group 'image_processors'

🔧 Worker A received:
  [1776555984275-0] {'task_id': 'task_0', 'type': 'process_image', 'url': 'img_0.jpg'}
  [1776555984275-1] {'task_id': 'task_1', 'type': 'process_image', 'url': 'img_1.jpg'}

🔧 Worker B received:
  [1776555984276-0] {'task_id': 'task_2', 'type': 'process_image', 'url': 'img_2.jpg'}
  [1776555984276-1] {'task_id': 'task_3', 'type': 'process_image', 'url': 'img_3.jpg'}

🔧 Worker C received:
  [1776555984277-0] {'task_id': 'task_4', 'type': 'process_image', 'url': 'img_4.jpg'}
  [1776555984277-1] {'task_id': 'task_5', 'type': 'process_image', 'url': 'img_5.jpg'}

💡 Each task was delivered to exactly ONE worker — no duplicates!


In [8]:
# Workers must acknowledge messages after processing them
# Unacknowledged messages can be reassigned if a worker crashes

# Check pending (unacknowledged) messages
pending = r.xpending("tasks", "image_processors")
print(f"📊 Pending summary: {pending}")
print(f"   Total unacknowledged: {pending['pending']}")

# Worker A acknowledges its tasks (simulating successful processing)
if worker_a_tasks:
    for stream, messages in worker_a_tasks:
        for msg_id, fields in messages:
            r.xack("tasks", "image_processors", msg_id)
            print(f"  ✅ Worker A acknowledged: {msg_id}")

# Check pending again
pending = r.xpending("tasks", "image_processors")
print(f"\n📊 After Worker A acknowledged: {pending['pending']} still pending")

print("\n💡 XACK tells Redis the message was processed successfully.")
print("   Unacknowledged messages can be reclaimed with XCLAIM if a worker dies.")

📊 Pending summary: {'pending': 6, 'min': '1776555984275-0', 'max': '1776555984277-1', 'consumers': [{'name': 'worker_a', 'pending': 2}, {'name': 'worker_b', 'pending': 2}, {'name': 'worker_c', 'pending': 2}]}
   Total unacknowledged: 6
  ✅ Worker A acknowledged: 1776555984275-0
  ✅ Worker A acknowledged: 1776555984275-1

📊 After Worker A acknowledged: 4 still pending

💡 XACK tells Redis the message was processed successfully.
   Unacknowledged messages can be reclaimed with XCLAIM if a worker dies.


### Recovering from Worker Failures

If a worker crashes without acknowledging, another worker can **claim** those messages after a timeout.

In [9]:
# Simulate: Worker B "crashed" without acknowledging its messages
# After some time, Worker A can claim those messages

# First, let's see what Worker B has pending
pending_detail = r.xpending_range("tasks", "image_processors", "-", "+", count=10)
print("📋 Pending messages by consumer:")
for p in pending_detail:
    print(f"  Message {p['message_id']} → {p['consumer']} (idle {p['time_since_delivered']}ms)")

# Claim Worker B's messages (messages idle for at least 0ms — in production, use a longer timeout)
# In production you'd use a higher min_idle_time (e.g., 30000ms = 30 seconds)
worker_b_pending = [p["message_id"] for p in pending_detail if p["consumer"] == "worker_b"]

if worker_b_pending:
    claimed = r.xclaim(
        "tasks", "image_processors", "worker_a",
        min_idle_time=0,  # 0ms for demo — use 30000+ in production
        message_ids=worker_b_pending
    )
    print(f"\n🔄 Worker A claimed {len(claimed)} messages from crashed Worker B:")
    for msg_id, fields in claimed:
        print(f"  [{msg_id}] {fields}")
        r.xack("tasks", "image_processors", msg_id)
        print(f"  ✅ Acknowledged: {msg_id}")

# Acknowledge worker C's messages too
if worker_c_tasks:
    for stream, messages in worker_c_tasks:
        for msg_id, fields in messages:
            r.xack("tasks", "image_processors", msg_id)

# Final check
pending = r.xpending("tasks", "image_processors")
print(f"\n📊 Final pending count: {pending['pending']}")
print("💡 All messages processed! XCLAIM is how Redis handles worker failures gracefully.")

📋 Pending messages by consumer:
  Message 1776555984276-0 → worker_b (idle 22ms)
  Message 1776555984276-1 → worker_b (idle 22ms)
  Message 1776555984277-0 → worker_c (idle 22ms)
  Message 1776555984277-1 → worker_c (idle 22ms)

🔄 Worker A claimed 2 messages from crashed Worker B:
  [1776555984276-0] {'task_id': 'task_2', 'type': 'process_image', 'url': 'img_2.jpg'}
  ✅ Acknowledged: 1776555984276-0
  [1776555984276-1] {'task_id': 'task_3', 'type': 'process_image', 'url': 'img_3.jpg'}
  ✅ Acknowledged: 1776555984276-1

📊 Final pending count: 0
💡 All messages processed! XCLAIM is how Redis handles worker failures gracefully.


## 4️⃣ When to Use What?

| Scenario | Use | Why |
|----------|-----|-----|
| Chat messages, live notifications | **Pub/Sub** | Real-time, ephemeral, low latency |
| Work queue with retries | **Streams + Consumer Groups** | Durability, acknowledgment, claiming |
| Event sourcing / audit log | **Streams** | Append-only, replayable |
| High-throughput event processing | **Kafka** | Designed for massive scale |
| Fan-out to many services | **Pub/Sub** (or SNS) | Simple broadcast |
| Exactly-once processing | **Kafka** | Redis gives at-least-once |

```
Need messaging?
├── Real-time, ephemeral? → Pub/Sub
├── Need durability + replay? → Streams
├── Need massive scale (>100K msg/s)? → Kafka
└── Need exactly-once? → Kafka with transactions
```

## 🧹 Cleanup

In [10]:
r.flushdb()
print("🧹 Cleaned up all keys from this notebook")

🧹 Cleaned up all keys from this notebook


## 📚 Summary

### Key Takeaways

1. **Pub/Sub** is fire-and-forget — perfect for real-time broadcasts, but messages are lost if no one is listening
2. **Streams** are durable append-only logs — messages persist and can be replayed
3. **Consumer Groups** divide work among multiple workers — each message goes to exactly one consumer
4. **XACK/XCLAIM** handle failures — unacknowledged messages can be reassigned to healthy workers
5. **Choose Pub/Sub** for ephemeral real-time communication (chat, notifications)
6. **Choose Streams** for durable event processing (work queues, audit logs)
7. **Choose Kafka** when you need massive throughput or exactly-once semantics

### Next Up

In **Notebook 3**, we'll explore **Redis as Cache vs Primary Store** — when to use Redis for caching vs as your main data store, plus patterns like distributed locks, rate limiting, and leaderboards.